# 重新在不同的筛选指标下利用AF3进行验证。将protenix的输入文件拆分成多个AF3的输入文件，存放在指定目录下，目录的名称为filter的名称，每个输入文件包含所有的蛋白-多肽复合物

In [1]:
import json
import os

In [2]:
def split_input_file(input_file, output_dir, af3_template):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    with open(input_file, 'r') as f:
        data = json.load(f)

    for job_data in data:
        job_name = job_data['name']
        pdb = job_name[:4]
        # print(pdb)
        pro_seq = job_data['sequences'][0]['proteinChain']['sequence']
        pro_msa = job_data['sequences'][0]['proteinChain']['msa']['precomputed_msa_dir']
        pep_seq = job_data['sequences'][1]['proteinChain']['sequence']

        output_file = os.path.join(output_dir, f'{job_name}.json')
        with open(af3_template, 'r') as template_f:
            template_data = json.load(template_f)

        template_data['name'] = job_name
        template_data['sequences'][0]['protein']['sequence'] = pro_seq
        template_data['sequences'][0]['protein']['pairedMsaPath'] = "/home/junjiechen/1_work/250401-Dpepalign/Benchmark/dataset_for_predict/MSA_pro_all_PepSet_dimer" + "/" + pdb + "/" + "pairing.a3m"
        template_data['sequences'][0]['protein']['unpairedMsaPath'] = "/home/junjiechen/1_work/250401-Dpepalign/Benchmark/dataset_for_predict/MSA_pro_all_PepSet_dimer" + "/" + pdb + "/" + "unpaired.a3m"

        template_data['sequences'][1]['protein']['sequence'] = pep_seq

        template_data['modelSeeds'] = [42, 43, 44]
        with open(output_file, 'w') as out_f:
            json.dump(template_data, out_f, indent=4)

In [4]:
for root, dirs, files in os.walk('/home/junjiechen/1_work/250401-Dpepalign/Benchmark/02_ligandmpnn/test_filter/inputs/Protenix'):
    for file in files:
        if file.endswith('.json') and file.startswith('pred_filter'):
            input_file = os.path.join(root, file)
            filter_name = file.split('.')[0]
            output_dir = os.path.join('/home/junjiechen/1_work/250401-Dpepalign/Benchmark/02_ligandmpnn/test_filter/inputs/AF3', filter_name)
            # os.makedirs(os.path.join('./AF3', filter_name, 'outputs'), exist_ok=True)
            split_input_file(input_file, output_dir, '/home/junjiechen/1_work/250401-Dpepalign/Benchmark/02_ligandmpnn/test_filter/scripts/AF3/template_af3_nomsa_notemplate.json')